# 22 — Part 7's share decomposition, corrected for recall of the 2020 vote

Notebook 11 splits Trump's margin among validated 2024 voters into net switching and new voters using **recalled** 2020 vote. Notebook 21 showed recall misclassifies. Here the CES 2020–24 panel supplies, for each recalled-2020-status × 2024-vote cell, the distribution of the 2020 status **reported in 2020**; the cross-section's cell shares are redistributed accordingly and the decomposition recomputed.

Fixed before estimation: the voter filter of notebook 11 (`TS_g2024` 1–6, 2024 vote known); primary truth = the 2020 report (`CC20_410`; no 2020 vote reported = non-voter); secondary = validated 2020 turnout (no voter-file record = non-voter), reported only as a bound because unmatched records are not all non-voters; intervals from 1,000 bootstrap resamples of the panel. Panelists had answered the item in 2020 and 2022, so their recall is better than a fresh respondent's and the correction is conservative.

In [1]:
pd_opts = None
import numpy as np, pandas as pd
ROOT = ".."
S20 = ["Biden", "Trump", "other", "nonvoter"]; V24 = ["Harris", "Trump", "third"]
v24 = lambda x: np.select([x == 1, x == 2, x.between(3, 7)], ["Harris", "Trump", "third"], None)
# cross-section joint table: recalled 2020 status x 2024 vote, weighted shares of validated 2024 voters
d = pd.read_csv(ROOT + r"\data\raw\ces\CCES24_Common_OUTPUT_vv_topost_final.csv", usecols=["TS_g2024", "presvote20post", "CC24_410", "vvweight_post", "vvweight"], low_memory=False)
d["r20"] = d.presvote20post.map({1: "Biden", 2: "Trump", 3: "other", 4: "other", 5: "other", 6: "nonvoter"}); d["v24"] = v24(d.CC24_410)
e = d[d.TS_g2024.notna() & (d.TS_g2024 != 7) & d.v24.notna() & d.r20.notna()].copy(); e["wt"] = e.vvweight_post.fillna(e.vvweight)
J = e.pivot_table(index="r20", columns="v24", values="wt", aggfunc="sum").reindex(index=S20, columns=V24).fillna(0); J /= J.values.sum()
# panel: P(true 2020 status | recalled status, 2024 vote), same voter filter
p = pd.read_csv(ROOT + r"\data\raw\ces_panel\merged_recontact_2024_vv.csv", usecols=["TS_g2024", "presvote20post_24", "CC24_410", "CC20_410", "CL_2020gvm", "commonpostweight_24", "commonweight_24"], low_memory=False)
p["r20"] = p.presvote20post_24.map({1: "Biden", 2: "Trump", 3: "other", 4: "other", 5: "other", 6: "nonvoter"}); p["v24"] = v24(p.CC24_410)
p["t20"] = p.CC20_410.map({1: "Biden", 2: "Trump", 4: "other", 5: "nonvoter", 6: "nonvoter"}); p.loc[p.CC20_410.isna(), "t20"] = "nonvoter"
p["t20_val"] = np.where(p.CL_2020gvm.isna(), "nonvoter", p.t20.replace({"nonvoter": "other"}))  # validated turnout: no record = nonvoter
p["wt"] = p.commonpostweight_24.fillna(p.commonweight_24)
q = p[p.TS_g2024.notna() & (p.TS_g2024 != 7) & p.v24.notna() & p.r20.notna() & p.t20.notna()].copy()

def corrected(q, tcol):
    T = pd.DataFrame(0.0, index=S20, columns=V24)
    for r in S20:
        for v in V24:
            c = q[(q.r20 == r) & (q.v24 == v)]
            if len(c) == 0 or J.loc[r, v] == 0: T.loc[r, v] += J.loc[r, v]; continue   # no panel cell: keep as recalled
            m = c.groupby(tcol).wt.sum() / c.wt.sum()
            for t, sh in m.items(): T.loc[t, v] += J.loc[r, v] * sh
    return T
def decomp(T):
    net = lambda s: 100 * (T.loc[s, "Trump"] - T.loc[s, "Harris"])
    return pd.Series({"switch_BT": 100 * T.loc["Biden", "Trump"], "switch_TH": 100 * T.loc["Trump", "Harris"],
                      "net_switching_pp": 100 * (T.loc["Biden", "Trump"] - T.loc["Trump", "Harris"]), "new_voters_pp": net("nonvoter"), "other_pp": net("other"),
                      "margin_pp": 100 * (T["Trump"].sum() - T["Harris"].sum()), "ratio_BT_TH": T.loc["Biden", "Trump"] / T.loc["Trump", "Harris"],
                      "biden_switch_rate": T.loc["Biden", "Trump"] / T.loc["Biden"].sum(), "new_share": T.loc["nonvoter"].sum(), "new_trump": T.loc["nonvoter", "Trump"] / T.loc["nonvoter"].sum()})


In [2]:
pd.set_option('display.width', 200)
def shares(s):
    tot = s.net_switching_pp + s.new_voters_pp + s.other_pp
    return pd.concat([s, pd.Series({'switch_share_of_shift': s.net_switching_pp / tot, 'new_share_of_shift': s.new_voters_pp / tot})])
out = pd.DataFrame({'recall (published)': shares(decomp(J)), 'corrected: 2020 report': shares(decomp(corrected(q, 't20'))),
                    'bound: validated 2020 turnout': shares(decomp(corrected(q, 't20_val')))})
rng = np.random.default_rng(22)
bs = pd.DataFrame([shares(decomp(corrected(q.sample(len(q), replace=True, random_state=rng.integers(1e9)), 't20'))) for _ in range(1000)])
out['corrected lo'] = bs.quantile(0.025); out['corrected hi'] = bs.quantile(0.975)
print(out.round(3).to_string()); out.to_csv('../outputs/tables/share_decomposition_recall_corrected.csv')
print('\npanel N by recalled status x 2024 vote:'); print(pd.crosstab(q.r20, q.v24))
for r, v in [('Biden', 'Trump'), ('nonvoter', 'Trump'), ('Trump', 'Harris'), ('nonvoter', 'Harris')]:
    print(f'recalled {r}, voted {v} in 2024 -> 2020 report:', q[(q.r20 == r) & (q.v24 == v)].t20.value_counts().to_dict())
print('panel 2024 voters reporting a 2020 vote but with no validated 2020 record: %.1f%%' % (100 * (q.CL_2020gvm.isna() & q.t20.isin(['Biden', 'Trump'])).mean()))

                       recall (published)  corrected: 2020 report  bound: validated 2020 turnout  corrected lo  corrected hi
switch_BT                           2.478                   1.922                          1.660         1.236         2.435
switch_TH                           0.734                   0.745                          0.656         0.606         0.874
net_switching_pp                    1.745                   1.177                          1.004         0.491         1.706
new_voters_pp                       0.984                   0.642                          2.346        -0.478         1.655
other_pp                            0.102                   0.386                         -0.315        -0.093         1.232
margin_pp                           1.693                   1.693                          1.693         1.693         1.693
ratio_BT_TH                         3.378                   2.581                          2.531         1.671         3.551
